# 🍃 Tea Leaf Disease Classification — Training Notebook

This notebook trains an **EfficientNet-B0** model using **transfer learning** to classify tea leaf images into three categories:

| Class | Description |
|-------|-------------|
| **Brown Blight** | Fungal disease causing brown spots |
| **Healthy Green Leaf** | No disease present |
| **Red Spider Mite** | Pest damage with rust discoloration |

---

**How to use this notebook:**
1. Upload your `DISEASE Dataset` folder (with the 3 sub-folders) to Google Drive or place it beside this notebook.
2. Run all cells from top to bottom.
3. The trained model will be saved to the `models/` folder.

> ⚡ **Tip:** Use a **GPU runtime** in Colab (`Runtime → Change runtime type → T4 GPU`) for much faster training.

## 1. Environment Setup

In [ ]:
# ============================================================
# 1a. Install dependencies (only needed on Colab)
# ============================================================
import importlib, subprocess, sys

REQUIRED = ['torch', 'torchvision', 'PIL', 'sklearn', 'tqdm', 'matplotlib', 'numpy']
for pkg in REQUIRED:
    try:
        importlib.import_module(pkg)
    except ImportError:
        real = {'PIL': 'Pillow', 'sklearn': 'scikit-learn'}.get(pkg, pkg)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', real])

print('✅ All dependencies are available.')

In [ ]:
# ============================================================
# 1b. Imports
# ============================================================
import os
import json
import random
from datetime import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from PIL import Image
from tqdm.auto import tqdm
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {"CUDA — " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 2. Mount Google Drive *(Colab only)*

If you're running locally or the dataset is already beside this notebook, **skip this cell**.

In [ ]:
# ============================================================
# 2. Mount Google Drive (run only in Colab)
# ============================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('✅ Google Drive mounted.')
except ImportError:
    IN_COLAB = False
    print('ℹ️  Not running in Colab — skipping Drive mount.')

## 3. Configuration

**Set `DATA_DIR` to point to your `DISEASE Dataset` folder.** All other hyperparameters can be tuned below.

In [ ]:
# ============================================================
# 3. Configuration — ✏️ EDIT PATHS HERE
# ============================================================

# ---------- Dataset path ----------
# Option A: Local / Jupyter — dataset next to this notebook
DATA_DIR = 'DISEASE Dataset'

# Option B: Google Colab — dataset in your Drive
# DATA_DIR = '/content/drive/MyDrive/Tharindu Project/DISEASE Dataset'

# ---------- Output directory ----------
SAVE_DIR = 'models'
os.makedirs(SAVE_DIR, exist_ok=True)

# ---------- Hyperparameters ----------
CONFIG = {
    'image_size':    224,
    'batch_size':    32,
    'learning_rate': 1e-4,
    'epochs':        50,
    'train_split':   0.8,
    'num_workers':   2 if IN_COLAB else 0,   # 0 for Windows
    'patience':      10,                      # early-stopping patience
}

CLASS_NAMES = ['Brown Blight', 'Healthy Green Leaf', 'Red spider Mite']

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('📋 Configuration')
for k, v in CONFIG.items():
    print(f'   {k:18s}: {v}')
print(f'   {"device":18s}: {DEVICE}')
print(f'   {"data_dir":18s}: {DATA_DIR}')
print(f'   {"save_dir":18s}: {SAVE_DIR}')

## 4. Data Loading & Augmentation

In [ ]:
# ============================================================
# 4a. Define transforms
# ============================================================
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(CONFIG['image_size']),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print('✅ Transforms defined.')

In [ ]:
# ============================================================
# 4b. Load dataset & split into train / val
# ============================================================
full_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transforms)

total_size = len(full_dataset)
train_size = int(CONFIG['train_split'] * total_size)
val_size   = total_size - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

# Apply val transforms to the validation split
val_dataset.dataset = datasets.ImageFolder(DATA_DIR, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                          shuffle=True,  num_workers=CONFIG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)

# ---- Class distribution ----
class_counts = {}
for _, label in full_dataset.samples:
    name = CLASS_NAMES[label]
    class_counts[name] = class_counts.get(name, 0) + 1

# Class weights for imbalanced data
total = sum(class_counts.values())
class_weights = torch.tensor([
    total / (len(CLASS_NAMES) * class_counts[c]) for c in CLASS_NAMES
], dtype=torch.float32)

print(f'📂 Dataset loaded from: {DATA_DIR}')
print(f'   Total images : {total_size}')
print(f'   Training     : {train_size} ({CONFIG["train_split"]*100:.0f}%)')
print(f'   Validation   : {val_size}  ({(1-CONFIG["train_split"])*100:.0f}%)')
print(f'\n📊 Class distribution:')
for c, n in class_counts.items():
    print(f'   {c:25s}: {n}')
print(f'\n⚖️  Class weights: {class_weights.tolist()}')

### Preview Sample Images

In [ ]:
# ============================================================
# 4c. Visualise a batch of training images
# ============================================================
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def denormalize(tensor):
    img = tensor.cpu().numpy().transpose(1, 2, 0)
    img = img * STD + MEAN
    return np.clip(img, 0, 1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, ax in enumerate(axes.flat):
    if i >= len(images):
        break
    ax.imshow(denormalize(images[i]))
    ax.set_title(CLASS_NAMES[labels[i]], fontsize=10)
    ax.axis('off')
fig.suptitle('Sample Training Images (augmented)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Model Architecture

In [ ]:
# ============================================================
# 5. Create EfficientNet-B0 with custom classifier
# ============================================================
def create_model(num_classes=3):
    """Load pre-trained EfficientNet-B0 and replace the classifier head."""
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

    # Freeze feature extractor
    for param in model.features.parameters():
        param.requires_grad = False

    # Custom classifier
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.2),
        nn.Linear(512, num_classes),
    )
    return model

model = create_model(num_classes=3).to(DEVICE)

# Quick summary
total_params   = sum(p.numel() for p in model.parameters())
trainable      = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'🧠 EfficientNet-B0 loaded')
print(f'   Total parameters     : {total_params:,}')
print(f'   Trainable parameters : {trainable:,}')
print(f'   Frozen parameters    : {total_params - trainable:,}')

## 6. Training Loop

In [ ]:
# ============================================================
# 6a. Helper functions: train_epoch & validate
# ============================================================
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc='  Train', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, preds = outputs.max(1)
        total   += labels.size(0)
        correct += preds.eq(labels).sum().item()
        pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{100.*correct/total:.1f}%')
    return running_loss / len(loader), 100. * correct / total


def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='  Val  ', leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, preds = outputs.max(1)
            total   += labels.size(0)
            correct += preds.eq(labels).sum().item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return running_loss / len(loader), 100. * correct / total, all_preds, all_labels

In [ ]:
# ============================================================
# 6b. Run training
# ============================================================
criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
patience_counter = 0

print('🚀 Starting training …')
print(f'   Epochs         : {CONFIG["epochs"]}')
print(f'   Batch size     : {CONFIG["batch_size"]}')
print(f'   Learning rate  : {CONFIG["learning_rate"]}')
print(f'   Early-stop     : patience {CONFIG["patience"]}\n')

for epoch in range(CONFIG['epochs']):
    print(f'Epoch [{epoch+1}/{CONFIG["epochs"]}]')

    # Unfreeze all layers after epoch 5 for fine-tuning
    if epoch == 5:
        print('   🔓 Unfreezing all layers for fine-tuning …')
        for param in model.parameters():
            param.requires_grad = True
        for pg in optimizer.param_groups:
            pg['lr'] = CONFIG['learning_rate'] * 0.1

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc, val_preds, val_labels = validate(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'   Train — Loss: {train_loss:.4f}  Acc: {train_acc:.2f}%')
    print(f'   Val   — Loss: {val_loss:.4f}  Acc: {val_acc:.2f}%')

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'class_names': CLASS_NAMES,
            'config': CONFIG,
        }, os.path.join(SAVE_DIR, 'best_model.pth'))
        print(f'   ✅ Saved best model (Val Acc: {val_acc:.2f}%)')
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['patience']:
            print(f'\n⚠️  Early stopping after {patience_counter} epochs without improvement.')
            break
    print()

print('=' * 60)
print(f'✅ Training complete!  Best Val Acc: {best_val_acc:.2f}%')
print('=' * 60)

## 7. Save Final Model

In [ ]:
# ============================================================
# 7. Save final model + training info
# ============================================================
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'class_names': CLASS_NAMES,
    'config': CONFIG,
}, os.path.join(SAVE_DIR, 'final_model.pth'))

with open(os.path.join(SAVE_DIR, 'training_info.json'), 'w') as f:
    json.dump({
        'best_val_acc': best_val_acc,
        'total_epochs': epoch + 1,
        'config': CONFIG,
        'class_names': CLASS_NAMES,
        'timestamp': datetime.now().isoformat(),
    }, f, indent=2)

print(f'💾 Saved to {SAVE_DIR}/')
print(f'   • best_model.pth')
print(f'   • final_model.pth')
print(f'   • training_info.json')

## 8. Training Curves

In [ ]:
# ============================================================
# 8. Plot loss & accuracy curves
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

# --- Loss ---
ax1.plot(epochs_range, history['train_loss'], label='Train Loss', color='#3498db', linewidth=2)
ax1.plot(epochs_range, history['val_loss'],   label='Val Loss',   color='#e74c3c', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Accuracy ---
ax2.plot(epochs_range, history['train_acc'], label='Train Acc', color='#3498db', linewidth=2)
ax2.plot(epochs_range, history['val_acc'],   label='Val Acc',   color='#e74c3c', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training & Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved training_history.png')

## 9. Evaluation on Validation Set

In [ ]:
# ============================================================
# 9a. Load best model & run final evaluation
# ============================================================
checkpoint = torch.load(os.path.join(SAVE_DIR, 'best_model.pth'), map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

_, final_acc, final_preds, final_labels = validate(model, val_loader, criterion, DEVICE)
print(f'\n🎯 Best Model — Validation Accuracy: {final_acc:.2f}%')

In [ ]:
# ============================================================
# 9b. Confusion Matrix
# ============================================================
cm = confusion_matrix(final_labels, final_preds)
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=True)
ax.set_title('Confusion Matrix', fontsize=14)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved confusion_matrix.png')

In [ ]:
# ============================================================
# 9c. Classification Report
# ============================================================
print('\n📋 Classification Report:\n')
print(classification_report(final_labels, final_preds, target_names=CLASS_NAMES))

## 10. Test Prediction on a Single Image

In [ ]:
# ============================================================
# 10. Predict a single image (change TEST_IMAGE_PATH as needed)
# ============================================================
def predict_single(model, image_path, device):
    """Run inference on one image and display the result."""
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(tensor)
        probs = F.softmax(out, dim=1)[0]

    pred_idx = probs.argmax().item()
    pred_class = CLASS_NAMES[pred_idx]
    confidence = probs[pred_idx].item() * 100

    # Display
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5),
                                    gridspec_kw={'width_ratios': [1, 1.3]})
    ax1.imshow(img)
    ax1.set_title(f'Prediction: {pred_class}\nConfidence: {confidence:.1f}%', fontsize=12)
    ax1.axis('off')

    colors = ['#e74c3c' if i != pred_idx else '#2ecc71' for i in range(len(CLASS_NAMES))]
    bars = ax2.barh(CLASS_NAMES, [p.item()*100 for p in probs], color=colors, edgecolor='white')
    ax2.set_xlabel('Probability (%)')
    ax2.set_xlim(0, 105)
    for bar, p in zip(bars, probs):
        ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{p.item()*100:.1f}%', va='center', fontsize=11)
    ax2.set_title('Class Probabilities', fontsize=12)
    plt.tight_layout()
    plt.show()

    return pred_class, confidence


# ---- Pick a random image from the dataset for testing ----
test_class = random.choice(os.listdir(DATA_DIR))
test_class_dir = os.path.join(DATA_DIR, test_class)
test_img_name = random.choice(os.listdir(test_class_dir))
TEST_IMAGE_PATH = os.path.join(test_class_dir, test_img_name)

print(f'🖼️  Testing with: {TEST_IMAGE_PATH}')
print(f'   (True class: {test_class})\n')
pred, conf = predict_single(model, TEST_IMAGE_PATH, DEVICE)

## 11. Download Model *(Colab only)*

Run this cell to download the trained model from Colab to your local machine.

In [ ]:
# ============================================================
# 11. Download trained model (Colab only)
# ============================================================
try:
    from google.colab import files
    files.download(os.path.join(SAVE_DIR, 'best_model.pth'))
    print('⬇️  Downloading best_model.pth …')
except ImportError:
    print('ℹ️  Not in Colab. Model is saved locally at:', os.path.join(SAVE_DIR, 'best_model.pth'))

---

### ✅ Done!

Your trained model files are saved in the `models/` folder:

| File | Description |
|------|-------------|
| `best_model.pth` | Best checkpoint (highest val accuracy) |
| `final_model.pth` | Last epoch checkpoint |
| `training_history.png` | Loss & accuracy curves |
| `confusion_matrix.png` | Confusion matrix plot |
| `training_info.json` | Training metadata |

Copy `best_model.pth` into your project's `models/` folder and run `python app.py` to start the web server.